<a href="https://colab.research.google.com/github/Prezzo-K/Graph-Neural-Networks-for-Fraud-Detection/blob/classical-ml-experiments/notebooks/Traditional_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import scikit learn and check version
import sklearn as sk

sk.__version__

'1.6.1'

In [ ]:
# Workflow
# 1. Get the dataset, do preprocessing if any and split it into train and test set
# 2. Load various clasical ML models - Random Forest (RF), Extra Trees, Logistic Regression (LR), Support Vector Machine (SVM), XGBoost
# 3. Train and test them on the test set
# 3. Compare their Accuracy, Precision and Recall, F1-Score, Area Under the ROC Curve (AUC–ROC), Confusion Matrix


# Load Dataset

In [ ]:
import pandas as pd

file_path = "/content/Fraud Detection Transactions Dataset.csv"

dataset  = pd.read_csv(filepath_or_buffer = file_path)
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Transaction_ID                50000 non-null  object 
 1   User_ID                       50000 non-null  object 
 2   Transaction_Amount            50000 non-null  float64
 3   Transaction_Type              50000 non-null  object 
 4   Timestamp                     50000 non-null  object 
 5   Account_Balance               50000 non-null  float64
 6   Device_Type                   50000 non-null  object 
 7   Location                      50000 non-null  object 
 8   Merchant_Category             50000 non-null  object 
 9   IP_Address_Flag               50000 non-null  int64  
 10  Previous_Fraudulent_Activity  50000 non-null  int64  
 11  Daily_Transaction_Count       50000 non-null  int64  
 12  Avg_Transaction_Amount_7d     50000 non-null  float64
 13  F

In [ ]:
# Retrieve the feature and labels separate

X = dataset.loc[:,"Transaction_ID":"Is_Weekend"]
y = dataset.loc[:, "Fraud_Label"]

X.head(2)

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,Location,Merchant_Category,IP_Address_Flag,Previous_Fraudulent_Activity,Daily_Transaction_Count,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend
0,TXN_33553,USER_1834,39.79,POS,2023-08-14 19:30:00,93213.17,Laptop,Sydney,Travel,0,0,7,437.63,3,Amex,65,883.17,Biometric,0.8494,0
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07 04:01:00,75725.25,Mobile,New York,Clothing,0,0,13,478.76,4,Mastercard,186,2203.36,Password,0.0959,0


In [ ]:
y.head(2), y.value_counts()

(0    0
 1    1
 Name: Fraud_Label, dtype: int64,
 Fraud_Label
 0    33933
 1    16067
 Name: count, dtype: int64)

In [ ]:
# Standardize columns with continous values - Transaction_Amount, Account_Balance, Daily_Transaction_Count, Avg_Transaction_Amount_7d, Card_Age, Transaction_Distance, Risk_Score
# Encode these as one hot encoding - Transaction_Type, Device_Type, Location, Merchant_Category, Card_Type, Authentication_Method
# Leave Binary / Numeric columns as they are. Don't scale or encode - IP_Address_Flag, Previous_Fraudulent_Activity, - Is_Weekend

# Drop the Transaction_ID, User_ID as they don't add in classification
X.drop(columns = ["Transaction_ID", "User_ID"], inplace = True)

# Convert 'Timestamp' to datetime and extract various fields
X["Timestamp"] = pd.to_datetime(X["Timestamp"])
X["Hour"] = X["Timestamp"].dt.hour
X["Day"] = X["Timestamp"].dt.day
X["Month"] = X["Timestamp"].dt.month
X["DayOfWeek"] = X["Timestamp"].dt.dayofweek
# drop off timestamp col now
X.drop(columns = ["Timestamp"], inplace = True)

# Do one hot encoding
categorical_cols = [
    "Transaction_Type",
    "Device_Type",
    "Location",
    "Merchant_Category",
    "Card_Type",
    "Authentication_Method"
]
X = pd.get_dummies(X, columns = categorical_cols)
# Display the first few rows with the new 'Hour' column to confirm
X

,Transaction_Amount,Account_Balance,IP_Address_Flag,Previous_Fraudulent_Activity,Daily_Transaction_Count,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Age,Transaction_Distance,Risk_Score,...,Merchant_Category_Restaurants,Merchant_Category_Travel,Card_Type_Amex,Card_Type_Discover,Card_Type_Mastercard,Card_Type_Visa,Authentication_Method_Biometric,Authentication_Method_OTP,Authentication_Method_PIN,Authentication_Method_Password
0,39.79,93213.17,0,0,7,437.63,3,65,883.17,0.8494,...,False,True,True,False,False,False,True,False,False,False
1,1.19,75725.25,0,0,13,478.76,4,186,2203.36,0.0959,...,False,False,False,False,True,False,False,False,False,True
2,28.96,1588.96,0,0,14,50.01,4,226,1909.29,0.8400,...,True,False,False,False,False,True,True,False,False,False
3,254.32,76807.20,0,0,8,182.48,4,76,1311.86,0.7935,...,False,False,False,False,False,True,False,True,False,False
4,31.28,92354.66,0,1,14,328.69,4,140,966.98,0.3819,...,False,False,False,False,True,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,45.05,76960.11,0,0,2,389.00,3,98,1537.54,0.1493,...,False,False,True,False,False,False,False,False,True,False
49996,126.15,28791.75,0,0,13,434.95,4,93,2555.72,0.3653,...,False,False,False,False,False,True,True,False,False,False
49997,72.02,29916.41,0,1,1,369.15,2,114,4686.59,0.5195,...,False,False,False,False,False,True,True,False,False,False
49998,64.89,67895.67,0,0,13,242.29,4,72,4886.92,0.7063,...,False,False,False,True,False,False,True,False,False,False


In [ ]:
# split the data into train and test set
from sklearn.model_selection import train_test_split

random_state = 42
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = test_size,
                                                    shuffle = True,
                                                    stratify = y
                                                    )
len(X_train), len(y_train), len(X_test), len(y_test)

(40000, 40000, 10000, 10000)

In [ ]:
# do some sanity check to ensure good splits of the classes in train and test sets
y_train.value_counts(), y_test.value_counts()

(Fraud_Label
 0    27146
 1    12854
 Name: count, dtype: int64,
 Fraud_Label
 0    6787
 1    3213
 Name: count, dtype: int64)

# Load Models And Train

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# create the model
rf = RandomForestClassifier()
# fit the model on the data
rf.fit(X_train, y_train)

RandomForestClassifier()

In [ ]:
preds = rf.predict(X_test)

In [ ]:
# metrics -  Accuracy, Precision and Recall, F1-Score, Area Under the ROC Curve (AUC–ROC), Confusion Matrix
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

accuracy = accuracy_score(preds, y_test)
f1 = f1_score(preds, y_test)
precison = precision_score(preds, y_test)
recall = recall_score(preds, y_test)
report = classification_report(preds, y_test)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"Precision: {precison:.4f}")
print(f"Recall: {recall:.4f}")
print("\nClassification Report:\n", report)


Accuracy: 1.0000
F1-Score: 1.0000
Precision: 1.0000
Recall: 1.0000

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      6787
           1       1.00      1.00      1.00      3213

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000

